# Music Annotation Project V5.3: Optimised Controlled Reconstruction Benchmark

**Runtime:** CPU.

This version corrects the prior benchmark:
- no reconstruction method receives the clean target;
- pixels outside the mask are immutable and asserted;
- data is staged on Colab local storage;
- reconstruction runs only on padded annotation crops;
- per-page results are cached;
- ground-truth features are computed once per page;
- blank, staff, symbol and mixed components are ranked with category-appropriate metrics;
- two biharmonic-based hybrids are tested;
- paired page deltas and bootstrap confidence intervals are exported.

The clean page is used only for evaluation after reconstruction.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas matplotlib


## Configuration and local staging


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

import cv2, json, hashlib, shutil, math, os
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
from skimage.restoration import inpaint as sk_inpaint

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_files(folder): return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if Path(folder).exists() else []
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def object_hash(value): return hashlib.md5(json.dumps(value,sort_keys=True).encode()).hexdigest()
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]
    assert np.array_equal(out[mask==0],original[mask==0])
    return out

def padded_components(mask,margin=40,min_area=8):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    boxes=[];h,w=mask.shape
    for i in range(1,n):
        x,y,bw,bh,a=stats[i]
        if a<min_area: continue
        boxes.append((max(0,x-margin),max(0,y-margin),min(w,x+bw+margin),min(h,y+bh+margin)))
    # merge overlapping padded rectangles
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            a=boxes.pop(); ax0,ay0,ax1,ay1=a; rest=[]
            for b in boxes:
                bx0,by0,bx1,by1=b
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
                    ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
                else: rest.append(b)
            boxes=rest;merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def crop_inpaint(image,mask,method,margin=40):
    result=image.copy()
    for x0,y0,x1,y1 in padded_components(mask,margin):
        im=image[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]
        if method=='telea': cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA)
        elif method=='navier_stokes': cand=cv2.inpaint(im,mk,5,cv2.INPAINT_NS)
        elif method=='biharmonic':
            z=sk_inpaint.inpaint_biharmonic(im.astype(np.float32)/255.0,mk>0,channel_axis=None)
            cand=np.clip(z*255,0,255).astype(np.uint8)
        else: raise ValueError(method)
        local=result[y0:y1,x0:x1];local[mk>0]=cand[mk>0];result[y0:y1,x0:x1]=local
    return immutable(image,result,mask)

def staff_evidence(observed,mask):
    visible=observed.copy();visible[mask>0]=255
    ink=(visible<190).astype(np.uint8)*255
    long=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))
    return cv2.dilate(long,np.ones((3,7),np.uint8))>0

def safe_staff_continuation(observed,mask,base,max_gap=80):
    out=base.copy(); evidence=staff_evidence(observed,mask); h,w=observed.shape
    rows=np.where(evidence.sum(axis=1)>=18)[0]
    for y in rows:
        xs=np.where(mask[y]>0)[0]
        if not xs.size: continue
        for start in np.split(xs,np.where(np.diff(xs)>1)[0]+1):
            if not len(start) or len(start)>max_gap: continue
            x0,x1=int(start[0]),int(start[-1]); r=16
            left=evidence[max(0,y-1):min(h,y+2),max(0,x0-r):x0]
            right=evidence[max(0,y-1):min(h,y+2),x1+1:min(w,x1+r+1)]
            if left.sum()<2 or right.sum()<2: continue
            vals=[]
            for yy in range(max(0,y-1),min(h,y+2)):
                for xa,xb in [(max(0,x0-r),x0),(x1+1,min(w,x1+r+1))]:
                    q=observed[yy,xa:xb]; vals.extend(q[q<190].tolist())
            if not vals: continue
            value=min(int(np.percentile(vals,35)),140)
            segment=(mask[max(0,y-1):min(h,y+2),x0:x1+1]>0)
            patch=out[max(0,y-1):min(h,y+2),x0:x1+1]
            patch[segment]=value
    return immutable(observed,out,mask)

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject');DRIVE_CLEAN=ROOT/'source'/'clean';DRIVE_MASK=ROOT/'masks'/'corrected';DRIVE_OUT=ROOT/'reconstruction_benchmark_v3'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S');OUT=DRIVE_OUT/RUN_NAME;LOCAL=Path('/content/reconstruction_benchmark_v3');LCLEAN=LOCAL/'clean';LMASK=LOCAL/'masks';CACHE=LOCAL/'cache'
for d in [OUT,OUT/'tables',OUT/'failure_crops',OUT/'mask_classes',OUT/'page_comparisons',LCLEAN,LMASK,CACHE]:d.mkdir(parents=True,exist_ok=True)
CROP_MARGIN=48;OCCLUSION_VALUE=50;MAX_FAILURE_CROPS=20;BOOTSTRAP_SAMPLES=2000;FORCE_REBUILD=False
METHODS=['telea','navier_stokes','biharmonic','hybrid_conservative','hybrid_routed']

def maps(folder):return {p.stem:p for p in image_files(folder)}
clean_map=maps(DRIVE_CLEAN);mask_map=maps(DRIVE_MASK);pages=sorted(set(clean_map)&set(mask_map))
if not pages:raise RuntimeError('No matched clean/mask pages.')
for page in pages:
    shutil.copy2(clean_map[page],LCLEAN/(page+clean_map[page].suffix.lower()));shutil.copy2(mask_map[page],LMASK/(page+mask_map[page].suffix.lower()))
print('Controlled pages:',len(pages),'Local staging:',LOCAL,'Durable output:',OUT)


## Region classification and reconstruction candidates


In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def classify(clean,mask):
    ink=printed_ink(clean);long=cv2.morphologyEx(ink.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long.astype(np.uint8),np.ones((3,7),np.uint8))>0;staff_near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0
    symbol=ink&~staff_near;m=mask>0
    return {'all':m,'background':m&~ink,'staff':m&staff,'symbol':m&symbol,'mixed':m&(staff_near)&ink},staff,symbol

def hybrid_conservative(observed,mask):
    base=crop_inpaint(observed,mask,'biharmonic',CROP_MARGIN)
    return safe_staff_continuation(observed,mask,base)

def hybrid_routed(observed,mask):
    bi=crop_inpaint(observed,mask,'biharmonic',CROP_MARGIN);ns=crop_inpaint(observed,mask,'navier_stokes',CROP_MARGIN)
    staff=staff_evidence(observed,mask);edge=cv2.dilate(cv2.Canny(observed,70,170),np.ones((3,3),np.uint8))>0
    routed=bi.copy();route=(mask>0)&edge&~staff;routed[route]=ns[route]
    return safe_staff_continuation(observed,mask,immutable(observed,routed,mask))

def reconstruct(observed,mask):
    return {'telea':crop_inpaint(observed,mask,'telea',CROP_MARGIN),'navier_stokes':crop_inpaint(observed,mask,'navier_stokes',CROP_MARGIN),'biharmonic':crop_inpaint(observed,mask,'biharmonic',CROP_MARGIN),'hybrid_conservative':hybrid_conservative(observed,mask),'hybrid_routed':hybrid_routed(observed,mask)}


## Metrics with not-applicable structural scores


In [ ]:
def edges(im):return cv2.Canny(im,70,170)>0
def f1(ref,pred,region,tol=1):
    ref=ref&region;pred=pred&region
    if ref.sum()==0:return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pd=cv2.dilate(pred.astype(np.uint8),k)>0
    precision=float(np.sum(pred&rd)/max(pred.sum(),1));recall=float(np.sum(ref&pd)/ref.sum());return precision,recall,2*precision*recall/max(precision+recall,1e-12)
def metrics(clean,recon,region):
    if region.sum()==0:return None
    diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32));mse=float(np.mean((clean[region].astype(float)-recon[region].astype(float))**2));_,sm=ssim(clean,recon,data_range=255,full=True)
    ip,ir,if1=f1(printed_ink(clean),printed_ink(recon),region,1);ep,er,ef1=f1(edges(clean),edges(recon),region,1)
    return {'pixels':int(region.sum()),'mae':float(diff[region].mean()),'psnr':float(20*np.log10(255/math.sqrt(max(mse,1e-9)))),'ssim':float(sm[region].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1_tolerant':ef1}
def component_category(regions,component):
    counts={k:int(np.sum(v&component)) for k,v in regions.items() if k!='all'};nonzero={k:v for k,v in counts.items() if v>0}
    if not nonzero:return 'background'
    top=max(nonzero,key=nonzero.get)
    if len([v for v in nonzero.values() if v>=.2*component.sum()])>1:return 'mixed'
    return top


## Cached controlled run


In [ ]:
rows=[];components=[];memory={}
for page in pages:
    cp=next(LCLEAN.glob(page+'.*'));mp=next(LMASK.glob(page+'.*'));clean=cv2.imread(str(cp),0);mask=cv2.imread(str(mp),0);mask=np.where(mask>0,255,0).astype(np.uint8)
    observed=clean.copy();observed[mask>0]=OCCLUSION_VALUE;regions,staff,symbol=classify(clean,mask)
    fingerprint=object_hash({'clean':md5(cp),'mask':md5(mp),'margin':CROP_MARGIN,'version':3});cache_path=CACHE/f'{page}_{fingerprint}.npz'
    if cache_path.exists() and not FORCE_REBUILD:
        z=np.load(cache_path);methods={m:z[m] for m in METHODS}
    else:
        methods=reconstruct(observed,mask);np.savez_compressed(cache_path,**methods)
    for m,r in methods.items():
        assert np.array_equal(r[mask==0],observed[mask==0]),f'{page} {m} changed unmasked pixels'
        for region_name,region in regions.items():
            q=metrics(clean,r,region)
            if q:rows.append({'page':page,'method':m,'region':region_name,'outside_mask_drift':0.0,**q})
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    for i in range(1,n):
        if stats[i,cv2.CC_STAT_AREA]<20:continue
        comp=lab==i;cat=component_category(regions,comp)
        for m,r in methods.items():
            q=metrics(clean,r,comp)
            if q:components.append({'page':page,'component':i,'category':cat,'method':m,'x':int(stats[i,0]),'y':int(stats[i,1]),'w':int(stats[i,2]),'h':int(stats[i,3]),**q})
    memory[page]=(clean,observed,mask,methods,regions);print('Benchmarked',page)
results=pd.DataFrame(rows);component_df=pd.DataFrame(components)
results.to_csv(OUT/'tables'/'metrics_by_page_region.csv',index=False);component_df.to_csv(OUT/'tables'/'component_metrics.csv',index=False)


## Summaries and paired bootstrap comparisons


In [ ]:
summary=results.groupby(['method','region'],as_index=False).agg(pages=('page','nunique'),pixels=('pixels','sum'),mae=('mae','mean'),psnr=('psnr','mean'),ssim=('ssim','mean'),ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),edge_f1_tolerant=('edge_f1_tolerant','mean'),outside_mask_drift=('outside_mask_drift','max'))
summary.to_csv(OUT/'tables'/'reconstruction_summary_by_region.csv',index=False);summary[summary.region=='all'].to_csv(OUT/'reconstruction_summary.csv',index=False)

def bootstrap_delta(values,seed=42):
    v=np.asarray(values,float);v=v[np.isfinite(v)]
    if not len(v):return (np.nan,np.nan,np.nan)
    rng=np.random.default_rng(seed);samples=np.array([np.mean(rng.choice(v,len(v),replace=True)) for _ in range(BOOTSTRAP_SAMPLES)])
    return float(np.mean(v)),float(np.percentile(samples,2.5)),float(np.percentile(samples,97.5))
paired=[]
base='biharmonic'
for region,metric,higher in [('background','mae',False),('staff','ink_recall',True),('symbol','edge_f1_tolerant',True),('mixed','edge_f1_tolerant',True),('all','ssim',True)]:
    sub=results[results.region==region].pivot(index='page',columns='method',values=metric)
    if base not in sub:continue
    for m in METHODS:
        if m==base or m not in sub:continue
        d=(sub[m]-sub[base])*(1 if higher else -1);mean,lo,hi=bootstrap_delta(d.dropna())
        paired.append({'region':region,'metric':metric,'method':m,'baseline':base,'benefit_delta_mean':mean,'ci95_low':lo,'ci95_high':hi,'wins':int((d>0).sum()),'losses':int((d<0).sum()),'ties':int((d==0).sum())})
pd.DataFrame(paired).to_csv(OUT/'tables'/'paired_method_deltas.csv',index=False);display(summary);display(pd.DataFrame(paired))


## Category-specific failure crops


In [ ]:
rank_rules={'background':('mae',False),'staff':('ink_recall',True),'symbol':('edge_f1_tolerant',True),'mixed':('edge_f1_tolerant',True)}
for category,(metric,higher) in rank_rules.items():
    target=component_df[(component_df.category==category)&(component_df.method=='hybrid_routed')].dropna(subset=[metric]).sort_values(metric,ascending=higher).head(MAX_FAILURE_CROPS)
    folder=OUT/'failure_crops'/category;folder.mkdir(parents=True,exist_ok=True)
    for rank,row in target.reset_index(drop=True).iterrows():
        clean,observed,mask,methods,regions=memory[row.page];H,W=clean.shape;x0=max(0,int(row.x)-28);y0=max(0,int(row.y)-28);x1=min(W,int(row.x+row.w)+28);y1=min(H,int(row.y+row.h)+28)
        panels=[clean[y0:y1,x0:x1],observed[y0:y1,x0:x1]]+[methods[m][y0:y1,x0:x1] for m in METHODS]
        names=['ground truth','occlusion']+METHODS;fig,axes=plt.subplots(1,len(panels),figsize=(4*len(panels),4))
        for ax,im,name in zip(axes,panels,names):ax.imshow(im,cmap='gray',vmin=0,vmax=255);ax.set_title(name);ax.axis('off')
        fig.suptitle(f'{category} | {row.page} component {int(row.component)} | {metric}={row[metric]:.3f}');plt.tight_layout();plt.savefig(folder/f'{rank+1:03d}_{row.page}_{int(row.component):03d}.png',dpi=170,bbox_inches='tight');plt.close(fig)
print('Failure crops:',OUT/'failure_crops')


## Copy cache and outputs


In [ ]:
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'pages':len(pages),'methods':METHODS,'guarantees':{'clean_used_only_for_scoring':True,'outside_mask_immutable':True,'crop_only_processing':True,'local_staging':True,'page_cache':True}}
(OUT/'benchmark_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Output:',OUT)
print('Key files:',OUT/'reconstruction_summary.csv',OUT/'tables'/'reconstruction_summary_by_region.csv',OUT/'tables'/'paired_method_deltas.csv')
